# Colab Processing: Training, Evaluation, & Deployment

Notebook ini didesain khusus untuk dijalankan secara **online** menggunakan GPU di **Google Colab**:
1. **Google Drive Mount**: Menghubungkan penyimpanan cloud Drive untuk membaca dataset hasil clean dan menyimpan model.
2. **K-Fold Training (MobileNetV2)**: Melakukan transfer learning dua tahap (Warm-up & Fine-tuning) dengan skema 5-Fold Cross Validation.
3. **Evaluasi**: Visualisasi kurva belajar, perbandingan akurasi antar fold, Confusion Matrix, Classification Report test set, dan analisis misklasifikasi.
4. **TFLite Deployment**: Konversi model terbaik ke Float32, Float16, dan Int8 TFLite, serta mengevaluasi ukuran dan performa test set masing-masing model.

## 1. Mount Google Drive & Setup Path

In [ ]:
import os
from google.colab import drive

# Hubungkan ke Google Drive Anda
drive.mount('/content/drive', force_remount=False)

# Jalur folder project Anda di Drive
BASE_PATH = '/content/drive/MyDrive/modeling'
TRAIN_ROOT = os.path.join(BASE_PATH, 'outputs', 'cleaned_data')
CLEANED_TESTING_ROOT = os.path.join(BASE_PATH, 'outputs', 'cleaned_testings')
ARTIFACTS_DIR = os.path.join(BASE_PATH, 'outputs', 'artifacts_mobilenetv2')
RESEARCH_VIS_DIR = os.path.join(ARTIFACTS_DIR, 'research_testing')
DEPLOYMENT_DIR = os.path.join(ARTIFACTS_DIR, 'tflite_deployment')

os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(RESEARCH_VIS_DIR, exist_ok=True)
os.makedirs(DEPLOYMENT_DIR, exist_ok=True)

print('📁 BASE_PATH:', BASE_PATH)
print('📁 TRAIN_ROOT:', TRAIN_ROOT)
print('📁 CLEANED_TESTING_ROOT:', CLEANED_TESTING_ROOT)
print('📁 ARTIFACTS_DIR:', ARTIFACTS_DIR)

## 2. Dependencies & Konfigurasi

In [ ]:
import sys
import json
import math
import random
import shutil
from pathlib import Path
import subprocess
import importlib

def ensure_module(import_name, pip_name=None):
    try:
        return importlib.import_module(import_name)
    except ModuleNotFoundError:
        pkg = pip_name or import_name
        print(f"Installing missing package: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        return importlib.import_module(import_name)

pd = ensure_module("pandas")
np = ensure_module("numpy")
tf = ensure_module("tensorflow")
ensure_module("matplotlib")
ensure_module("sklearn", "scikit-learn")

from matplotlib import pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow:", tf.__version__)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

WARMUP_EPOCHS = 6
WARMUP_LR = 1e-3
FINETUNE_EPOCHS = 4
FINETUNE_LR = 1e-5
N_SPLITS = 5

## 3. Class Mapping & Data Loaders

In [ ]:
def normalize_class_name(type_name, category_name):
    return f"{type_name}_{category_name}".strip().lower().replace(" ", "_")

def collect_paths_and_labels(train_root):
    paths = []
    labels = []
    class_names = []
    top_types = [d for d in sorted(os.listdir(train_root)) if os.path.isdir(os.path.join(train_root, d))]
    for type_name in top_types:
        type_dir = os.path.join(train_root, type_name)
        categories = [d for d in sorted(os.listdir(type_dir)) if os.path.isdir(os.path.join(type_dir, d))]
        for category in categories:
            class_name = normalize_class_name(type_name, category)
            class_idx = len(class_names)
            class_names.append(class_name)

            class_dir = os.path.join(type_dir, category)
            for fname in os.listdir(class_dir):
                fpath = os.path.join(class_dir, fname)
                if os.path.isfile(fpath) and fname.lower().endswith((".jpg", ".jpeg", ".png")):
                    paths.append(fpath)
                    labels.append(class_idx)
    return np.array(paths), np.array(labels), class_names

paths, labels, CLASS_NAMES = collect_paths_and_labels(TRAIN_ROOT)
NUM_CLASSES = len(CLASS_NAMES)

print("Total training samples:", len(paths))
print("Num classes:", NUM_CLASSES)
print("Class names:", CLASS_NAMES)

def decode_and_resize(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.image.decode_image(image_bytes, channels=3, expand_animations=False)
    image = tf.image.resize(image, IMG_SIZE, method=tf.image.ResizeMethod.BILINEAR)
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    label_oh = tf.one_hot(label, depth=NUM_CLASSES)
    return image, label_oh

def make_dataset(paths_array, labels_array, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths_array, labels_array))
    if training:
        ds = ds.shuffle(buffer_size=len(paths_array), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_and_resize, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

## 4. Pelatihan Model MobileNetV2 (Warm-up & Fine-tune)

In [ ]:
def build_model():
    base = MobileNetV2(include_top=False, weights="imagenet", input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    inputs = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="classifier")(x)
    model = models.Model(inputs, outputs, name="MobileNetV2_Colab")
    return model, base

def unfreeze_upper_layers(base_model, trainable_fraction=0.35):
    total_layers = len(base_model.layers)
    cutoff = int(total_layers * (1.0 - trainable_fraction))
    for index, layer in enumerate(base_model.layers):
        layer.trainable = index >= cutoff

def combine_history(history_one, history_two, key):
    return history_one.history.get(key, []) + history_two.history.get(key, [])

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_results = []
all_fold_histories = {}
best_fold_acc = -1.0
best_fold_model_path = None
best_fold_history = None
best_fold_number = None

for fold_number, (train_index, val_index) in enumerate(skf.split(paths, labels), start=1):
    print(f"\n{'=' * 70}\nFold {fold_number}/{N_SPLITS}\n{'=' * 70}")
    tf.keras.backend.clear_session()
    model, base = build_model()

    train_ds = make_dataset(paths[train_index], labels[train_index], training=True)
    val_ds = make_dataset(paths[val_index], labels[val_index], training=False)

    fold_dir = os.path.join(ARTIFACTS_DIR, f"fold_{fold_number}")
    os.makedirs(fold_dir, exist_ok=True)
    fold_ckpt = os.path.join(fold_dir, "best_fold.keras")

    callbacks = [
        EarlyStopping(monitor="val_accuracy", patience=4, mode="max", restore_best_weights=True),
        ModelCheckpoint(fold_ckpt, monitor="val_accuracy", mode="max", save_best_only=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1),
    ]

    # Phase 1: Warm-up
    print("[Phase 1] Warm-up (base frozen)")
    base.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=WARMUP_LR), loss="categorical_crossentropy", metrics=["accuracy"])
    hist_warmup = model.fit(train_ds, validation_data=val_ds, epochs=WARMUP_EPOCHS, callbacks=callbacks, verbose=1)

    # Phase 2: Fine-tuning
    print("[Phase 2] Fine-tuning (upper layers unfrozen)")
    unfreeze_upper_layers(base, trainable_fraction=0.35)
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization): layer.trainable = False

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=FINETUNE_LR), loss="categorical_crossentropy", metrics=["accuracy"])
    hist_finetune = model.fit(train_ds, validation_data=val_ds, epochs=FINETUNE_EPOCHS, callbacks=callbacks, verbose=1)

    all_fold_histories[str(fold_number)] = {
        "accuracy": combine_history(hist_warmup, hist_finetune, "accuracy"),
        "val_accuracy": combine_history(hist_warmup, hist_finetune, "val_accuracy"),
        "loss": combine_history(hist_warmup, hist_finetune, "loss"),
        "val_loss": combine_history(hist_warmup, hist_finetune, "val_loss"),
    }

    best_model = tf.keras.models.load_model(fold_ckpt)
    fold_loss, fold_acc = best_model.evaluate(val_ds, verbose=0)
    print(f"✅ Fold {fold_number} best val accuracy: {fold_acc:.4f} | val loss: {fold_loss:.4f}")

    fold_results.append({
        "fold": fold_number,
        "val_accuracy": float(fold_acc),
        "val_loss": float(fold_loss),
        "train_size": int(len(train_index)),
        "val_size": int(len(val_index)),
        "fold_model_path": fold_ckpt,
    })

    if fold_acc > best_fold_acc:
        best_fold_acc = fold_acc
        best_fold_model_path = os.path.join(ARTIFACTS_DIR, "best_kfold_model.keras")
        best_model.save(best_fold_model_path)
        best_fold_history = all_fold_histories[str(fold_number)]
        best_fold_number = fold_number

In [ ]:
# Simpan ringkasan hasil K-Fold
results_df = pd.DataFrame(fold_results)
histories_json_path = os.path.join(ARTIFACTS_DIR, "kfold_fold_histories.json")
mean_val_acc = float(results_df["val_accuracy"].mean())
std_val_acc = float(results_df["val_accuracy"].std(ddof=0))
mean_val_loss = float(results_df["val_loss"].mean())
std_val_loss = float(results_df["val_loss"].std(ddof=0))

summary = {
    "train_root": TRAIN_ROOT,
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "num_samples": int(len(paths)),
    "n_splits": N_SPLITS,
    "fold_results": fold_results,
    "mean_val_accuracy": mean_val_acc,
    "std_val_accuracy": std_val_acc,
    "mean_val_loss": mean_val_loss,
    "std_val_loss": std_val_loss,
    "best_fold": best_fold_number,
    "best_fold_val_accuracy": float(best_fold_acc),
    "best_model_path": best_fold_model_path,
}

summary_path = os.path.join(ARTIFACTS_DIR, "kfold_training_summary.json")
results_csv_path = os.path.join(ARTIFACTS_DIR, "kfold_results.csv")

with open(summary_path, "w", encoding="utf-8") as f: json.dump(summary, f, indent=2)
with open(histories_json_path, "w", encoding="utf-8") as f: json.dump(all_fold_histories, f, indent=2)
results_df.to_csv(results_csv_path, index=False)

print("Mean val accuracy:", mean_val_acc)
print("Best fold:", best_fold_number)

## 5. Visualisasi Metrik Training & Evaluasi Fold

In [ ]:
# Plot Accuracy & Loss per Epoch untuk Fold terbaik
epochs = np.arange(1, len(best_fold_history["accuracy"]) + 1)
fig, ax1 = plt.subplots(figsize=(12, 6.8))
line1 = ax1.plot(epochs, best_fold_history["accuracy"], label="Train Accuracy", color="#1d3557", linewidth=2)
line2 = ax1.plot(epochs, best_fold_history["val_accuracy"], label="Val Accuracy", color="#457b9d", linewidth=2, linestyle="--")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.set_ylim(0, 1.05)
ax1.grid(True, linestyle="--", alpha=0.3)

ax2 = ax1.twinx()
line3 = ax2.plot(epochs, best_fold_history["loss"], label="Train Loss", color="#e76f51", linewidth=2)
line4 = ax2.plot(epochs, best_fold_history["val_loss"], label="Val Loss", color="#f4a261", linewidth=2, linestyle="--")
ax2.set_ylabel("Loss")

lines = line1 + line2 + line3 + line4
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="lower center", bbox_to_anchor=(0.5, -0.18), ncol=4)
plt.title(f"Training Metrics (Fold {best_fold_number})", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, f"fold_{best_fold_number}_time_series.png"), bbox_inches="tight")
plt.show()

In [ ]:
# Plot perbandingan akurasi dan loss antar fold
plt.figure(figsize=(10, 4))
plt.bar(results_df["fold"].astype(str), results_df["val_accuracy"], color="#457b9d")
plt.axhline(mean_val_acc, color="#e63946", linestyle="--", linewidth=2, label=f"Mean val acc = {mean_val_acc:.4f}")
plt.title("Validation Accuracy per Fold")
plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)
plt.grid(axis="y", linestyle="--", alpha=0.25)
plt.legend()
plt.savefig(os.path.join(ARTIFACTS_DIR, "fold_validation_accuracy_comparison.png"), bbox_inches="tight")
plt.show()

## 6. Pengujian Test Set (Evaluasi Performa)

In [ ]:
def collect_all_image_files(root_folder):
    image_files = []
    for current_root, _, files in os.walk(root_folder):
        for file_name in files:
            if file_name.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                image_files.append(os.path.join(current_root, file_name))
    return image_files

def collect_split_paths_and_labels(root_dir, class_names):
    label_map = {name: idx for idx, name in enumerate(class_names)}
    paths_out, labels_out = [], []
    if not os.path.exists(root_dir): return np.array(paths_out), np.array(labels_out)
    for type_folder in sorted(os.listdir(root_dir)):
        type_path = os.path.join(root_dir, type_folder)
        if not os.path.isdir(type_path): continue
        for category_folder in sorted(os.listdir(type_path)):
            category_path = os.path.join(type_path, category_folder)
            if not os.path.isdir(category_path): continue
            class_name = normalize_class_name(type_folder, category_folder)
            if class_name not in label_map: continue
            class_idx = label_map[class_name]
            for image_path in collect_all_image_files(category_path):
                paths_out.append(image_path)
                labels_out.append(class_idx)
    return np.array(paths_out), np.array(labels_out)

def make_inference_dataset(paths_array, labels_array):
    ds = tf.data.Dataset.from_tensor_slices((paths_array, labels_array))
    ds = ds.map(decode_and_resize, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

def plot_confusion_matrix(cm, class_names, output_path, title):
    plt.figure(figsize=(12, 10))
    plt.imshow(cm, interpolation="nearest", cmap="Blues")
    plt.title(title, fontweight="bold")
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45, ha="right")
    plt.yticks(tick_marks, class_names)
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    threshold = cm.max() / 2.0 if cm.size else 0.0
    for r in range(cm.shape[0]):
        for c in range(cm.shape[1]):
            plt.text(c, r, format(cm[r, c], "d"), ha="center", va="center",
                     color="white" if cm[r, c] > threshold else "black", fontsize=8)
    plt.tight_layout()
    plt.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close()

def plot_misclassified_contact_sheet(misclassified_rows, output_path, page_index, page_size=12):
    if not misclassified_rows: return
    rows = max(1, math.ceil(len(misclassified_rows) / 4))
    fig, axes = plt.subplots(rows, 4, figsize=(16, 4 * rows))
    axes = np.atleast_1d(axes).flatten()
    for ax in axes[len(misclassified_rows):]: ax.axis("off")
    for ax, row in zip(axes, misclassified_rows):
        try:
            image_bytes = tf.io.read_file(row["path"])
            image = tf.image.decode_image(image_bytes, channels=3, expand_animations=False)
            ax.imshow(image.numpy().astype(np.uint8))
            ax.set_title(f"True: {row['true_class']}\nPred: {row['pred_class']}\nConf: {row['confidence']:.3f}", fontsize=8)
            ax.axis("off")
        except:
            ax.axis("off")
    plt.suptitle(f"Misclassified Images - Page {page_index}", y=1.02, fontweight="bold")
    plt.tight_layout()
    plt.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close()

In [ ]:
# Jalankan Evaluasi pada data testing
test_paths, test_labels = collect_split_paths_and_labels(CLEANED_TESTING_ROOT, CLASS_NAMES)
if len(test_paths) == 0:
    print("❌ Dataset testing bersih tidak ditemukan di outputs/cleaned_testings/!")
else:
    test_ds = make_inference_dataset(test_paths, test_labels)
    best_model = tf.keras.models.load_model(best_fold_model_path)
    test_loss, test_accuracy = best_model.evaluate(test_ds, verbose=0)
    probabilities = best_model.predict(test_ds, verbose=0)
    y_pred = np.argmax(probabilities, axis=1)
    y_true = test_labels[:len(y_pred)]
    confidences = probabilities.max(axis=1)

    # Save report
    report_dict = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    pd.DataFrame(report_dict).transpose().to_csv(os.path.join(RESEARCH_VIS_DIR, "testing_classification_report.csv"))
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))
    
    # Confusion matrix
    plot_confusion_matrix(confusion_matrix(y_true, y_pred), CLASS_NAMES, os.path.join(RESEARCH_VIS_DIR, "testing_confusion_matrix.png"), "Testing Confusion Matrix")

    # Extract error analysis
    misclassified = []
    misclassified_dir = os.path.join(RESEARCH_VIS_DIR, "misclassified_images")
    if os.path.exists(misclassified_dir): shutil.rmtree(misclassified_dir)
    os.makedirs(misclassified_dir, exist_ok=True)
    
    for idx, (path, t_lbl, p_lbl, conf) in enumerate(zip(test_paths, y_true, y_pred, confidences), start=1):
        if int(t_lbl) != int(p_lbl):
            row = {"path": str(path), "true_class": CLASS_NAMES[int(t_lbl)], "pred_class": CLASS_NAMES[int(p_lbl)], "confidence": float(conf)}
            misclassified.append(row)
            shutil.copy2(path, os.path.join(misclassified_dir, f"err_{idx:04d}_true-{row['true_class']}_pred-{row['pred_class']}_conf-{conf:.3f}.jpg"))
            
    pd.DataFrame(misclassified).to_csv(os.path.join(RESEARCH_VIS_DIR, "misclassified_index.csv"), index=False)
    print(f"✅ Akurasi Test Set: {test_accuracy*100:.2f}%")
    print(f"✅ Total Misclassified: {len(misclassified)}")
    
    page_size = 12
    total_pages = max(1, math.ceil(len(misclassified) / page_size)) if misclassified else 0
    for page in range(total_pages):
        plot_misclassified_contact_sheet(misclassified[page*page_size : (page+1)*page_size], os.path.join(RESEARCH_VIS_DIR, f"misclassified_page_{page + 1}.png"), page + 1)

## 7. Kuantisasi & Deployment TensorFlow Lite (TFLite)

In [ ]:
# Generator Dataset Representatif untuk Kuantisasi INT8
def representative_dataset_generator():
    train_paths, _ = collect_split_paths_and_labels(TRAIN_ROOT, CLASS_NAMES)
    if len(train_paths) == 0: return
    sample_count = min(100, len(train_paths))
    indices = np.linspace(0, len(train_paths) - 1, sample_count, dtype=int)
    for index in indices:
        image_path = train_paths[index]
        image_bytes = tf.io.read_file(image_path)
        image = tf.image.decode_image(image_bytes, channels=3, expand_animations=False)
        image = tf.image.resize(image, IMG_SIZE, method=tf.image.ResizeMethod.BILINEAR)
        image = tf.cast(image, tf.float32)
        image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
        yield [tf.expand_dims(image, axis=0)]

def save_bytes(path, content):
    with open(path, "wb") as f: f.write(content)

def convert_to_json_serializable(obj):
    if isinstance(obj, np.ndarray): return obj.tolist()
    elif isinstance(obj, np.integer): return int(obj)
    elif isinstance(obj, np.floating):
        if np.isnan(obj): return None
        return float(obj)
    elif isinstance(obj, float):
        if math.isnan(obj): return None
        return obj
    elif isinstance(obj, dict): return {str(key): convert_to_json_serializable(value) for key, value in obj.items()}
    elif isinstance(obj, list): return [convert_to_json_serializable(value) for value in obj]
    elif isinstance(obj, tuple): return [convert_to_json_serializable(value) for value in obj]
    else: return obj

In [ ]:
# Jalankan Konversi TFLite
model_keras = tf.keras.models.load_model(best_fold_model_path)

# FP32
converter_fp32 = tf.lite.TFLiteConverter.from_keras_model(model_keras)
tflite_fp32 = converter_fp32.convert()
fp32_path = os.path.join(DEPLOYMENT_DIR, "mobilenetv2_float32.tflite")
save_bytes(fp32_path, tflite_fp32)

# FP16
converter_fp16 = tf.lite.TFLiteConverter.from_keras_model(model_keras)
converter_fp16.optimizations = [tf.lite.Optimize.DEFAULT]
converter_fp16.target_spec.supported_types = [tf.float16]
tflite_fp16 = converter_fp16.convert()
fp16_path = os.path.join(DEPLOYMENT_DIR, "mobilenetv2_float16.tflite")
save_bytes(fp16_path, tflite_fp16)

# INT8
converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model_keras)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset_generator
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type = tf.int8
converter_int8.inference_output_type = tf.int8
tflite_int8 = converter_int8.convert()
int8_path = os.path.join(DEPLOYMENT_DIR, "mobilenetv2_int8.tflite")
save_bytes(int8_path, tflite_int8)
print("✅ Semua varian model TFLite berhasil di-convert!")

In [ ]:
# Helper Evaluasi Model TFLite pada Test Set
def evaluate_tflite_model(tflite_path, dataset_paths, dataset_labels):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    input_index, output_index, input_dtype = input_details["index"], output_details["index"], input_details["dtype"]
    input_scale, input_zero_point = input_details.get("quantization", (0.0, 0))
    output_scale, output_zero_point = output_details.get("quantization", (0.0, 0))

    predictions, trues = [], []
    for image_path, true_label in zip(dataset_paths, dataset_labels):
        image_bytes = tf.io.read_file(image_path)
        image = tf.image.decode_image(image_bytes, channels=3, expand_animations=False)
        image = tf.image.resize(image, IMG_SIZE, method=tf.image.ResizeMethod.BILINEAR)
        image = tf.cast(image, tf.float32)
        image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
        sample = tf.expand_dims(image, axis=0).numpy()

        if input_dtype in (np.int8, np.uint8):
            if input_scale and input_zero_point is not None:
                sample = np.round(sample / input_scale + input_zero_point)
            sample = np.clip(sample, np.iinfo(input_dtype).min, np.iinfo(input_dtype).max).astype(input_dtype)
        else:
            sample = sample.astype(input_dtype)

        interpreter.set_tensor(input_index, sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_index)

        if output.dtype in (np.int8, np.uint8) and output_scale:
            output = (output.astype(np.float32) - output_zero_point) * output_scale
        predictions.append(int(np.argmax(output[0])))
        trues.append(int(true_label))
    predictions = np.array(predictions)
    trues = np.array(trues)
    accuracy = float(np.mean(predictions == trues))
    
    # Dapatkan classification report dalam format dict dan str (text)
    report_dict = classification_report(trues, predictions, target_names=CLASS_NAMES, zero_division=0, output_dict=True)
    report_str = classification_report(trues, predictions, target_names=CLASS_NAMES, zero_division=0, output_dict=False, digits=3)
    cm = confusion_matrix(trues, predictions)
    
    return accuracy, report_dict, report_str, cm

if len(test_paths) > 0:
    exported_models = [
        {"name": "float32", "path": fp32_path, "size_mb": float(os.path.getsize(fp32_path) / (1024 * 1024))},
        {"name": "float16", "path": fp16_path, "size_mb": float(os.path.getsize(fp16_path) / (1024 * 1024))},
        {"name": "int8", "path": int8_path, "size_mb": float(os.path.getsize(int8_path) / (1024 * 1024))},
    ]
    evaluation_rows = []
    for item in exported_models:
        print(f"\n=========================================")
        print(f"📊 Evaluasi Model TFLite: {item['name'].upper()}")
        print(f"=========================================")
        acc, report_dict, report_str, cm = evaluate_tflite_model(item["path"], test_paths, test_labels)
        item["accuracy"] = float(acc)
        item["report"] = report_dict
        item["cm"] = cm
        evaluation_rows.append({
            "format": item["name"],
            "size_mb": float(item["size_mb"]),
            "test_accuracy": float(acc),
            "test_loss": None
        })
        
        # 1. Print classification report lengkap ke output notebook
        print("\nClassification Report:")
        print(report_str)
        
        # 2. Save Classification Report ke file CSV
        report_df = pd.DataFrame(report_dict).transpose()
        report_csv_path = os.path.join(DEPLOYMENT_DIR, f"classification_report_{item['name']}.csv")
        report_df.to_csv(report_csv_path)
        print(f"💾 Report disimpan ke: {report_csv_path}")
        
        # 3. Save Confusion Matrix ke file CSV
        cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
        cm_csv_path = os.path.join(DEPLOYMENT_DIR, f"confusion_matrix_{item['name']}.csv")
        cm_df.to_csv(cm_csv_path)
        print(f"💾 Confusion Matrix (CSV) disimpan ke: {cm_csv_path}")
        
        # 4. Plot & Save Confusion Matrix ke file PNG
        cm_png_path = os.path.join(DEPLOYMENT_DIR, f"confusion_matrix_{item['name']}.png")
        plot_confusion_matrix(cm, CLASS_NAMES, cm_png_path, f"Confusion Matrix - {item['name'].upper()}")
        print(f"🖼️ Confusion Matrix (Plot) disimpan ke: {cm_png_path}\n")
        
    eval_df = pd.DataFrame(evaluation_rows)
    eval_df.to_csv(os.path.join(DEPLOYMENT_DIR, "tflite_evaluation_summary.csv"), index=False)
    
    # Plot Model Size Comparison
    plt.figure(figsize=(8, 4))
    plt.bar(eval_df["format"], eval_df["size_mb"], color=["#1d3557", "#457b9d", "#e76f51"])
    plt.title("Perbandingan Ukuran Model TFLite")
    plt.ylabel("Ukuran (MB)")
    for idx, val in enumerate(eval_df["size_mb"]):
        plt.text(idx, val, f"{val:.2f} MB", ha="center", va="bottom")
    plt.savefig(os.path.join(DEPLOYMENT_DIR, "tflite_size_comparison.png"), dpi=200)
    plt.show()
    
    # Plot Model Accuracy Comparison
    plt.figure(figsize=(8, 4))
    plt.bar(eval_df["format"], eval_df["test_accuracy"], color=["#264653", "#2a9d8f", "#e9c46a"])
    plt.title("Perbandingan Akurasi Model TFLite")
    plt.ylabel("Akurasi")
    plt.ylim(0, 1.05)
    for idx, val in enumerate(eval_df["test_accuracy"]):
        plt.text(idx, val, f"{val:.4f}", ha="center", va="bottom")
    plt.savefig(os.path.join(DEPLOYMENT_DIR, "tflite_accuracy_comparison.png"), dpi=200)
    plt.show()
    
    # Simpan JSON Summary akhir
    summary_data = {
        "best_model_path": best_fold_model_path,
        "deployment_dir": DEPLOYMENT_DIR,
        "class_names": CLASS_NAMES,
        "total_testing_images": int(len(test_paths)),
        "exported_models": exported_models,
        "evaluation": evaluation_rows
    }
    with open(os.path.join(DEPLOYMENT_DIR, "tflite_deployment_summary.json"), "w", encoding="utf-8") as f:
        json.dump(convert_to_json_serializable(summary_data), f, indent=2, ensure_ascii=False)
    print("✅ Deployment TFLite lengkap & Evaluasi selesai!")